# Trimmed Meta-Topic ↔ Entity Graph — Gephi Export

A readable companion to the full all-NER divergence graph: instead of every
entity, this keeps only two curated sets per meta-topic centre —

- **distinctive** entities (the fans): top-N per topic by TF-IDF, so they show
  up strongly in one topic and barely elsewhere
- **shared** entities (the centre): top-M globally by how many topics they
  span, so they sit in the middle of the force layout

Only **news** and **talkshows** have `topic_meta` — kamer is excluded, same as
the earlier metatopic divergence graph.

**Cleaning** reuses the stage-one pipeline: normalisation, noise filter
(empty/single-char/numeric), alias merge via the review workbook's `canonical`
column for `keep="yes"` rows, and drop everything `keep="no"` plus the small
junk stop-list.

In [12]:
import ast
import re
import math
import pathlib
import numpy as np
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS
# ============================================================

SOURCES = {
    "news": {
        "path":     "../../news/analysis/df_with_NER.csv",
        "topic_col": "topic",
        "meta_col":  "topic_meta",
    },
    "talkshows": {
        "path":     "../../subtitles/analysis/subs_with_NER.csv",
        "topic_col": "topic",
        "meta_col":  "topic_meta",
    },
}

ENTITY_COLS = {
    "persons":   "PER",
    "orgs":      "ORG",
    "countries": "LOC",
}

REVIEW_XLSX   = "entity_review.xlsx"        # three-sheet workbook (PER/ORG/LOC); optional
JUNK_STOPLIST = {"wie", "dat.", "anders"}    # normalised lowercase; always dropped

TOP_N_DISTINCTIVE = 15     # top entities per topic by TF-IDF
TOP_M_SHARED       = 50    # top entities globally by topic spread
MIN_TOPIC_FREQ      = 3    # drop (entity, topic) pairs below this before ranking
MIN_EDGE_WEIGHT      = 0.01  # coverage-share floor for drawing an edge

OUT_NODES        = "metatopic_trim_nodes.csv"
OUT_EDGES        = "metatopic_trim_edges.csv"
OUT_TOPICS_XLSX  = "metatopic_trim_topics.xlsx"
# ============================================================

## 1. Helpers

In [13]:
def parse_entity_list(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    return re.sub(r"\s+", " ", str(s).strip())


def slugify(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def make_unique_id(prefix, label, used_ids):
    base = slugify(label) or "unk"
    candidate = f"{prefix}_{base}"
    if candidate not in used_ids:
        used_ids.add(candidate)
        return candidate
    i = 2
    while f"{candidate}_{i}" in used_ids:
        i += 1
    final = f"{candidate}_{i}"
    used_ids.add(final)
    return final

## 2. Alias/drop lookup from review workbook (if present)

In [14]:
entity_to_canonical = {}
canonical_to_type_review = {}
drop_set = set(JUNK_STOPLIST)

review_path = (NB_DIR / REVIEW_XLSX).resolve()
if review_path.exists():
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        for _, row in sheet_df.iterrows():
            key = normalise(row["entity"]).lower()
            if str(row["keep"]).strip().lower() == "no":
                drop_set.add(key)
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        kept = sheet_df[sheet_df["keep"].astype(str).str.strip().str.lower() == "yes"]
        for _, row in kept.iterrows():
            key = normalise(row["entity"]).lower()
            if key in drop_set:
                continue
            canonical = normalise(row["canonical"])
            entity_to_canonical[key] = canonical
            canonical_to_type_review[canonical] = sheet
    print(f"Review workbook found: {len(entity_to_canonical)} alias mappings, "
          f"{len(drop_set)} dropped keys (incl. junk stop-list)")
else:
    print("No review workbook found — only junk stop-list applied.")

Review workbook found: 239 alias mappings, 61 dropped keys (incl. junk stop-list)


## 3. Load news + talkshows (only sources with `topic_meta`)

In [15]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found")
        continue
    df = pd.read_csv(abs_path)
    df = df[df[cfg["topic_col"]] != -1].copy()
    df[cfg["meta_col"]] = df[cfg["meta_col"]].fillna("UNKNOWN").astype(str).str.strip()
    frames[arena] = df
    print(f"[OK] {arena}: {len(df):,} docs, {df[cfg['meta_col']].nunique()} meta-topics")

[OK] news: 13,209 docs, 18 meta-topics
[OK] talkshows: 495 docs, 12 meta-topics


## 4. Parse all documents → long format

In [16]:
records = []
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df = frames[arena]
    meta_col = cfg["meta_col"]
    for row_idx, row in df.iterrows():
        topic_meta = row[meta_col]
        doc_id = f"{arena}:{row_idx}"
        doc_seen = {}   # canonical -> raw_type (first seen, deduped within doc)
        for col, raw_type in ENTITY_COLS.items():
            for raw in parse_entity_list(row.get(col)):
                norm = normalise(raw)
                if len(norm) <= 1 or norm.isdigit():
                    continue
                key = norm.lower()
                if key in drop_set:
                    continue
                canonical = entity_to_canonical.get(key, norm)
                if canonical not in doc_seen:
                    doc_seen[canonical] = raw_type
        for canonical, raw_type in doc_seen.items():
            records.append({
                "topic_meta": topic_meta,
                "doc_id":     doc_id,
                "canonical":  canonical,
                "raw_type":   raw_type,
            })

long_df = pd.DataFrame(records)
print(f"(doc, canonical) pairings: {len(long_df):,}")
print(f"Distinct canonical entities: {long_df['canonical'].nunique():,}")

(doc, canonical) pairings: 194,371
Distinct canonical entities: 74,396


## 5. Resolve entity_type, topic doc totals, and the freq table

In [17]:
# entity_type: review-corrected wins, else doc-level majority vote
type_doc = long_df[["canonical", "doc_id", "raw_type"]].drop_duplicates()
type_counts = type_doc.groupby(["canonical", "raw_type"]).size().reset_index(name="n")
computed_dominant_type = (
    type_counts.loc[type_counts.groupby("canonical")["n"].idxmax()]
    .set_index("canonical")["raw_type"]
)
canonical_type = {
    c: canonical_to_type_review.get(c, computed_dominant_type.get(c, ""))
    for c in long_df["canonical"].unique()
}

# total documents per meta-topic (coverage-share denominator)
topic_doc_totals = {}
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    for meta, grp in frames[arena].groupby(cfg["meta_col"]):
        topic_doc_totals[meta] = topic_doc_totals.get(meta, 0) + len(grp)
total_topics = len(topic_doc_totals)
print(f"Total meta-topics: {total_topics}")

# freq_full: distinct docs per (topic, entity) — used later for edge weights (no floor)
freq_full = long_df.groupby(["topic_meta", "canonical"])["doc_id"].nunique()

# freq_filtered: candidate pool for TF-IDF / shared ranking
freq_filtered = freq_full[freq_full >= MIN_TOPIC_FREQ]
print(f"(topic, entity) pairs: {len(freq_full):,} total, "
      f"{len(freq_filtered):,} after min_topic_freq={MIN_TOPIC_FREQ}")

Total meta-topics: 18
(topic, entity) pairs: 111,985 total, 9,766 after min_topic_freq=3


## 6. TF-IDF and entity selection

- **distinctive**: top `top_n_distinctive` per topic by `tf * idf`
- **shared**: top `top_m_shared` globally by topic spread (tie-break: total freq)
- on overlap, **shared wins** (no duplicate node)

In [18]:
topics_per_entity = freq_filtered.groupby("canonical").size()
idf = np.log(total_topics / topics_per_entity)

tfidf_df = freq_filtered.reset_index()
tfidf_df.columns = ["topic_meta", "canonical", "freq"]
tfidf_df["idf"]   = tfidf_df["canonical"].map(idf)
tfidf_df["tfidf"] = tfidf_df["freq"] * tfidf_df["idf"]

# --- distinctive: top N per topic ---
distinctive_pairs = (
    tfidf_df.sort_values(["topic_meta", "tfidf"], ascending=[True, False])
    .groupby("topic_meta")
    .head(TOP_N_DISTINCTIVE)
    .copy()
)
distinctive_entities = set(distinctive_pairs["canonical"])

# --- shared: top M globally by topic spread, tie-break total freq ---
entity_stats = (
    tfidf_df.groupby("canonical")
    .agg(n_topics=("topic_meta", "nunique"), total_freq=("freq", "sum"))
    .sort_values(["n_topics", "total_freq"], ascending=[False, False])
)
shared_entities = set(entity_stats.head(TOP_M_SHARED).index)

# --- combine, shared takes precedence ---
final_roles = {e: "distinctive" for e in distinctive_entities}
final_roles.update({e: "shared" for e in shared_entities})

n_shared = sum(1 for v in final_roles.values() if v == "shared")
n_distinctive = sum(1 for v in final_roles.values() if v == "distinctive")
print(f"Distinctive entities (pure): {n_distinctive}")
print(f"Shared entities: {n_shared}")
print(f"Total selected entities: {len(final_roles)}")

Distinctive entities (pure): 247
Shared entities: 50
Total selected entities: 297


## 7. Build nodes

In [19]:
entity_total_docs = long_df.groupby("canonical")["doc_id"].nunique()

# best-scoring topic per distinctive entity (for dominant_topic on overlap)
best_topic_for_entity = (
    distinctive_pairs.loc[distinctive_pairs.groupby("canonical")["tfidf"].idxmax()]
    .set_index("canonical")["topic_meta"]
)

used_ids = set()
topic_node_rows = [
    {
        "Id":             make_unique_id("meta", t, used_ids),
        "Label":          t,
        "node_type":      "metatopic",
        "entity_type":    "",
        "role":           "",
        "size":           total,
        "dominant_topic": t,
    }
    for t, total in topic_doc_totals.items()
]
topic_id = {r["Label"]: r["Id"] for r in topic_node_rows}

entity_node_rows = [
    {
        "Id":             make_unique_id("ent", e, used_ids),
        "Label":          e,
        "node_type":      "entity",
        "entity_type":    canonical_type.get(e, ""),
        "role":           role,
        "size":           int(entity_total_docs.get(e, 0)),
        "dominant_topic": "shared" if role == "shared" else best_topic_for_entity.get(e, ""),
    }
    for e, role in final_roles.items()
]
entity_id = {r["Label"]: r["Id"] for r in entity_node_rows}

nodes_df = pd.DataFrame(topic_node_rows + entity_node_rows)
dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
if len(dupes):
    print(f"WARNING: duplicate node Ids: {dupes['Id'].tolist()[:10]}")
else:
    print("Node Ids: no collisions")
print(f"Meta-topic nodes: {len(topic_node_rows)}, entity nodes: {len(entity_node_rows)}")

Node Ids: no collisions
Meta-topic nodes: 18, entity nodes: 297


## 8. Build edges

For each selected entity, edges go to **every** topic where its coverage share
(using `freq_full`, not the `min_topic_freq`-filtered table) exceeds
`min_edge_weight`. Distinctive entities will mostly produce one edge; shared
entities will produce many.

In [20]:
edge_rows = []
for e in final_roles:
    for topic, total in topic_doc_totals.items():
        f = freq_full.get((topic, e), 0)
        if f == 0:
            continue
        share = f / total
        if share <= MIN_EDGE_WEIGHT:
            continue
        edge_rows.append({
            "Source": topic_id[topic],
            "Target": entity_id[e],
            "Weight": round(share, 6),
            "Type":   "Undirected",
        })

edges_df = pd.DataFrame(edge_rows)
print(f"Edges: {len(edges_df)}")

Edges: 976


## 9. Export

In [21]:
nodes_path = NB_DIR / OUT_NODES
edges_path = NB_DIR / OUT_EDGES
nodes_df.to_csv(nodes_path, index=False)
edges_df.to_csv(edges_path, index=False)

# per-topic readable table: the raw top-N TF-IDF ranking per topic
per_topic = distinctive_pairs.copy()
per_topic["entity_type"] = per_topic["canonical"].map(canonical_type)
per_topic["final_role"]  = per_topic["canonical"].map(final_roles)
per_topic = per_topic.rename(columns={
    "topic_meta": "meta_topic", "canonical": "entity",
    "freq": "freq_in_topic", "tfidf": "tfidf_score",
})
per_topic = per_topic[["meta_topic", "entity", "entity_type", "freq_in_topic", "tfidf_score", "final_role"]]
per_topic["tfidf_score"] = per_topic["tfidf_score"].round(3)

topics_path = NB_DIR / OUT_TOPICS_XLSX
per_topic.to_excel(topics_path, index=False)

print(f"Saved -> {nodes_path.name}, {edges_path.name}, {topics_path.name}")

Saved -> metatopic_trim_nodes.csv, metatopic_trim_edges.csv, metatopic_trim_topics.xlsx


## 10. Summary

In [22]:
entity_degree = edges_df.groupby("Target")["Source"].nunique()

print("=" * 55)
print("SUMMARY")
print(f"  Distinctive entities (pure): {n_distinctive}")
print(f"  Shared entities:             {n_shared}")
print(f"  Meta-topic nodes:            {len(topic_node_rows)}")
print(f"  Total edges:                 {len(edges_df)}")
print()

id_to_label = dict(zip(nodes_df["Id"], nodes_df["Label"]))
shared_ids = {entity_id[e] for e in shared_entities}
distinctive_only_ids = {entity_id[e] for e in final_roles if final_roles[e] == "distinctive"}

print("Top 5 shared entities (highest degree):")
for eid, deg in entity_degree[entity_degree.index.isin(shared_ids)].sort_values(ascending=False).head(5).items():
    print(f"  {id_to_label[eid]:<25s} degree={deg}")
print()
print("Sample distinctive entities (degree=1, the 'fans'):")
for eid, deg in entity_degree[entity_degree.index.isin(distinctive_only_ids)].sort_values().head(5).items():
    print(f"  {id_to_label[eid]:<25s} degree={deg}")
print("=" * 55)

SUMMARY
  Distinctive entities (pure): 247
  Shared entities:             50
  Meta-topic nodes:            18
  Total edges:                 976

Top 5 shared entities (highest degree):
  Japan                     degree=18
  Google                    degree=18
  Nederland                 degree=18
  NRC                       degree=18
  Verenigde Staten          degree=18

Sample distinctive entities (degree=1, the 'fans'):
  AEX                       degree=1
  Afas                      degree=1
  AFM                       degree=1
  Agentschap Telecom        degree=1
  AI-camera                 degree=1
